In [37]:
!pip install -q gradio groq

In [ ]:
import os
import gradio as gr
from groq import Groq


from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API")


client = Groq()


def respond(message: str, history: list, system_prompt: str, temperature: float):
    """
    Streams a reply from Groq and yields incremental text so
    Gradio can display it token-by-token.

    Parameters
    ----------
    message       : latest user message (str)
    history       : list of [user, assistant] pairs maintained by ChatInterface
    system_prompt : editable system prompt from the textbox
    temperature   : float 0-2 controlling randomness
    """


    messages = [{"role": "system", "content": system_prompt}]


    for user_msg, assistant_msg in history:
        messages.append({"role": "user",      "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})


    messages.append({"role": "user", "content": message})


    stream = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=messages,
        temperature=temperature,
        max_tokens=1024,
        stream=True,
    )


    partial_reply = ""
    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        partial_reply += delta
        yield partial_reply



system_prompt_box = gr.Textbox(
    value="You are a helpful and friendly AI assistant.",
    label=" System Prompt",
    placeholder="Describe the personality / role you want the bot to play…",
    lines=3,
)

temperature_slider = gr.Slider(
    minimum=0.0,
    maximum=2.0,
    value=0.7,
    step=0.05,
    label=" Temperature  (0 = deterministic · 2 = very creative)",
)

demo = gr.ChatInterface(
    fn=respond,
    title="Chatbot",

    additional_inputs=[system_prompt_box, temperature_slider],
    additional_inputs_accordion=gr.Accordion(
        label="  Chatbot Settings", open=True
    ),
    examples=[
        ["Tell me a short joke."],
        ["Explain quantum entanglement in simple terms."],
        ["Write a haiku about monsoon rain."],
    ],
    cache_examples=False,
)


demo.launch(share=True, debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://13a0783c11c6903dac.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
